
# SkinAI Eye-Tracking Data Analysis Notebook

This notebook analyzes eye-tracking data from SkinAI experiments, processing fixation data and generating visualizations.

Experiments:
- jpsychSkinAI

## Prerequisites & Setup

### 1. Environment Setup
- **Python Environment**: Use Python 3.7+ with Jupyter Notebook or VS Code
- **Required Libraries**: Install dependencies using `pip install -r requirements.txt`

### 2. Virtual Environment (Recommended)
```bash
# Create virtual environment
python -m venv skinai_env

# Activate (Windows)
skinai_env\Scripts\activate.bat

# Activate (macOS/Linux)
source skinai_env/bin/activate

# Install dependencies
pip install -r requirements.txt
```

### 3. Data Structure
Ensure your data follows this structure:
```
YourDataPath/
├── data/
│   ├── participant1/
│   │   ├── participant1_record.csv (or *_record_extra.csv)
│   │   └── participant1_metadata.csv
│   ├── participant2/
│   │   └── ...
│   └── ...
└── analysis/ (created automatically)
```

### 4. Configuration
- **Update the data path** in Cell 3 to point to your data directory
- **Adjust PADDING variable** if needed (default: 0 pixels). Examples:
  - `PADDING = 10` - Same 10px padding on all sides for all boxes
  - `PADDING = [20, 10]` - 20px horizontal, 10px vertical padding for all boxes
  - `PADDING = {'center': [5, 5], 'AI_box': [15, 10], 'slider_box': [0, 20]}` - Different padding per box
- **Set max_event_duration**: Change `max_event_duration=10000` in the `handle_carryover_fixations_and_merge()` function call to match your experiment's maximum trial duration (in milliseconds)

## What This Notebook Does

### 1. Data Processing Pipeline
1. **Directory Setup**: Creates analysis folder for output
2. **Participant Processing**: Iterates through each participant folder
3. **Data Loading**: Reads fixation data from CSV files
4. **Event Filtering**: Focuses on 'target_on' events
5. **Metadata Addition**: Adds participant ID, image dimensions, and padding

### 2. Bounding Box Management
- **Center Box**: Image dimensions (always included)
- **AI Box**: From `AIBoxCoord` column (if available)
- **Slider Box**: From `sliderBoxCoord` column (if available)

### 3. Analysis Functions
- **`plot2d()`**: Creates scatter plots with raw gaze data and fixations
- **`getFixationLatency()`**: Calculates fixation onset relative to target presentation
- **`handle_carryover_fixations_and_merge()`**: Handles fixations spanning event boundaries
- **`addAOI()`**: Assigns fixations to Areas of Interest (bounding boxes)

### 4. Output Generation
- **Individual plots**: Saved for each trial
- **Combined dataset**: `allSubjects_SkinAI.csv` with all participants' data

## Output Files (in /analysis folder)
1. **Trial plots**: Individual visualization files for each trial
2. **Combined data**: `allSubjects_SkinAI.csv` with processed fixation data
3. **Social indices**: Calculated metrics (if applicable)

## Key Data Columns

### Raw Data
- `user_pred_px_x, user_pred_px_y`: Raw gaze coordinates (pixels)
- `FixXPos, FixYPos`: Fixation positions (pixels)
- `FixDur`: Fixation duration (milliseconds)

### Processing Results
- `FixLatency`: Time from target onset to fixation start (ms)
- `FixationOrder`: Sequential order of fixations within trial
- `AOI_bbox`: Bounding box number where fixation landed (or None)
- `AOI_stim`: Stimulus identifier where fixation occurred

### Helper Variables
- `DistFromPrevFix`: Distance from previous fixation (pixels)
- `PrevFixXPos, PrevFixYPos`: Previous fixation coordinates
- `PrevFixSampTime`: Timestamp of previous fixation
- `FixStartEnd`: Indicates if fixation spans event boundaries

## Customization Options

### Adjusting Areas of Interest (AOIs)
- **Default**: AOIs match image dimensions
- **Flexible Padding**: The `PADDING` variable now supports multiple formats:
  - **Single value**: `PADDING = 10` (same padding on all sides for all boxes)
  - **Horizontal/Vertical**: `PADDING = [20, 10]` (20px horizontal, 10px vertical for all boxes)
  - **Per-box specific**: `PADDING = {'center': [5, 5], 'AI_box': [15, 10], 'slider_box': [0, 20]}`
- **Custom Boxes**: Add new bounding boxes by modifying the `create_bboxes()` function

### Plot Titles
- **Single condition**: `condition='diagnose'`
- **Multiple conditions**: `condition=['diagnose', 'aiOutput']`

## Troubleshooting
- **No debug output**: Check that `plot2d()` function is being called
- **Missing columns**: Verify your CSV files contain required columns
- **Path errors**: Ensure data path in Cell 3 is correct and accessible
- **Incorrect trial durations**: Adjust `max_event_duration` parameter to match your experiment's maximum trial length (default: 10000ms = 10 seconds)

In [4]:
import os
import pandas as pd
import numpy as np

# import DeepEye analysis functions
from deepeye_analysis_package.preprocessing import getFixationLatency, handle_carryover_fixations_and_merge, addAOI
from deepeye_analysis_package.plotting import plot2d
from deepeye_analysis_package.getFixations import extract_fixations

### Set the main path

In [5]:
# path = './CollectedData/Approved'
# path = r'C:/Users/aby600/Dropbox/Appliedwork/CognitiveSolutions/Projects/DeepEye/TechnicalReports/TechnicalReport1/Test_ASDViewing/ASD/Approved'
# path = r'D:/Dropbox/Appliedwork/CognitiveSolutions/Projects/DeepEye/TechnicalReports/TechnicalReport1/Test_ASDViewing/ASD/Approved'
path = r'C:/Users/artem/Dropbox/Appliedwork/CognitiveSolutions/Projects/DeepEye/TechnicalReports/TechnicalReport1/Test_SkinAI/Approved'


### Main part

In [6]:
# Define the AOI padding in pixels
PADDING = {
       'center': [0, 0],      # 5px horizontal, 5px vertical for center box
       'AI_box': [0, 50],    # 15px horizontal, 10px vertical for AI box
       'slider_box': [10, 50]  # No horizontal padding, 20px vertical for slider box
   }

# Helper function to create a directory if it doesn't exist
def create_directory_if_not_exists(directory_path):
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
        print(f"Directory '{directory_path}' was created.")
    else:
        print(f"Directory '{directory_path}' already exists.")

# Define data analysis directories and create them if they don't exist yet
path_to_data = os.path.join(path, 'data')
path_to_analysis = os.path.join(path, 'analysis')
create_directory_if_not_exists(path_to_analysis)

# Initialize an empty list to hold the processed dataframes
output_dfs = []

# Get all folder names from the data directory
folder_names = [name for name in os.listdir(path_to_data) if os.path.isdir(os.path.join(path_to_data, name))]

# Process each participant's data
for fn in folder_names:
    # Check if _record_extra.csv exists, if not use _record.csv
    path_to_file = os.path.join(path_to_data, fn, f'{fn}_record_extra.csv')
    if not os.path.exists(path_to_file):
        path_to_file = os.path.join(path_to_data, fn, f'{fn}_record.csv')

        print(f'Extracting fixations for participant {fn}...')

        try:
            df = pd.read_csv(path_to_file, on_bad_lines='skip')
            df = extract_fixations(df, path_to_file)
            df = df.drop_duplicates(subset=['user_pred_px_x', 'user_pred_px_y'], ignore_index=True)  # Ensure unique coordinates (sometimes webcam is stuck)

        except FileNotFoundError:
            print(f'File does not exist: {path_to_file}')
            continue    
    else: 
        try:
            df = pd.read_csv(path_to_file, on_bad_lines='skip')
            df = df.drop_duplicates(subset=['user_pred_px_x', 'user_pred_px_y'], ignore_index=True)  # Ensure unique coordinates (sometimes webcam is stuck)

        except FileNotFoundError:
            print(f'File does not exist: {path_to_file}')
            continue

    print(f'Processing participant {fn}...')

    # Filter data to only include rows where the target was presented
    df1 = df[df['event'] == 'target_on'].copy()

    # Add padding, subject ID, and image dimensions to the dataframe
    df1['padding'] = PADDING
    df1['deepeye_id'] = fn
    
    # Check if imgSize column exists and use it, otherwise use default dimensions
    if 'imgSize' in df1.columns:
        # Extract dimensions from imgSize column (format: "[500,500]" as string)
        def parse_img_size(img_size_str):
            if pd.isna(img_size_str):
                raise ValueError("imgSize is NaN")
            import ast
            dimensions = ast.literal_eval(img_size_str)
            if not isinstance(dimensions, list) or len(dimensions) != 2:
                raise ValueError(f"Expected [width, height] format, got: {img_size_str}")
            return (int(dimensions[0]), int(dimensions[1]))
        
        df1['imageDims'] = df1['imgSize'].apply(parse_img_size)
        print(f"Using image dimensions from imgSize column: {df1['imageDims'].iloc[0]}")
    else:
        # Fallback to hardcoded dimensions
        df1['imageDims'] = [(500, 500)] * len(df1)
        print("imgSize column not found, using default dimensions (500, 500)")

    ### Convert critical columns to numeric values if they are not ###
    df1['X'] = pd.to_numeric(df1['X'], errors='coerce')   
    df1['Y'] = pd.to_numeric(df1['Y'], errors='coerce')
    df1['FixXPos'] = pd.to_numeric(df1['FixXPos'], errors='coerce')
    df1['FixYPos'] = pd.to_numeric(df1['FixYPos'], errors='coerce')    
    df1['resX'] = pd.to_numeric(df1['resX'], errors='coerce')
    df1['resY'] = pd.to_numeric(df1['resY'], errors='coerce')
    df1['scrW_cm'] = pd.to_numeric(df1['scrW_cm'], errors='coerce')
    df1['PrevFixXPos'] = pd.to_numeric(df1['PrevFixXPos'], errors='coerce')
    df1['PrevFixYPos'] = pd.to_numeric(df1['PrevFixYPos'], errors='coerce')
    df1['PrevFixSampTime'] = pd.to_numeric(df1['PrevFixSampTime'], errors='coerce')
    df1['DistFromPrevFix'] = pd.to_numeric(df1['DistFromPrevFix'], errors='coerce')

   # Add image paths and coordinates to the dataframe
    df1['image_paths'] = df1['imageName'].apply(lambda name: [ name[1:] if name.startswith('/') else name ]) ## Remove the first slash character if present    
        
    df1['image_coords'] = df1.apply(lambda row: [
        (row.X, row.Y, row.imageDims[0], row.imageDims[1])        
    ], axis=1)

    # Add bounding boxes and their names to the dataframe
    def create_bboxes(row):
        import ast
        bboxes = [[row.X, row.Y, row.imageDims[0], row.imageDims[1]]]  # center bbox
        
        # Add AI box if it exists
        if pd.notna(row.get('AIBoxCoord')) and row.AIBoxCoord:
            # Parse string to dictionary if needed
            if isinstance(row.AIBoxCoord, str):
                ai_box = ast.literal_eval(row.AIBoxCoord)
            else:
                ai_box = row.AIBoxCoord
            bboxes.append([ai_box['x'], ai_box['y'], ai_box['width'], ai_box['height']])
        
        # Add slider box if it exists
        if pd.notna(row.get('sliderBoxCoord')) and row.sliderBoxCoord:
            # Parse string to dictionary if needed
            if isinstance(row.sliderBoxCoord, str):
                slider_box = ast.literal_eval(row.sliderBoxCoord)
            else:
                slider_box = row.sliderBoxCoord
            bboxes.append([slider_box['x'], slider_box['y'], slider_box['width'], slider_box['height']])
        
        return bboxes
    
    def create_bbox_names(row):
        names = ['center']
        
        # Add AI box name if it exists
        if pd.notna(row.get('AIBoxCoord')) and row.AIBoxCoord:
            names.append('AI_box')
        
        # Add slider box name if it exists
        if pd.notna(row.get('sliderBoxCoord')) and row.sliderBoxCoord:
            names.append('slider_box')
        
        return names
    
    df1['bboxes'] = df1.apply(create_bboxes, axis=1)
    df1['bboxesNames'] = df1.apply(create_bbox_names, axis=1)


    # Plot 2D fixations without saving the plot
    plot2d(df1, fn, path_to_analysis, condition=['diagnose', 'aiOutput'], save=True, padding=PADDING)

    # Process the data by applying preprocessing steps
    df1 = getFixationLatency(df1)
    df1 = handle_carryover_fixations_and_merge(df1, max_event_duration=10000)
    df1 = addAOI(df1, padding=PADDING)

    # Accumulate the processed dataframe for this participant
    output_dfs.append(df1)

# Concatenate all participants' data into one DataFrame
if output_dfs:
    output_df = pd.concat(output_dfs, ignore_index=True)
    output_file = os.path.join(path_to_analysis, 'allSubjects_SkinAI.csv')
    output_df.to_csv(output_file, index=False)
    print(f'Combined data saved to {output_file}')
else:
    print('No data was processed.')


Directory 'C:/Users/artem/Dropbox/Appliedwork/CognitiveSolutions/Projects/DeepEye/TechnicalReports/TechnicalReport1/Test_SkinAI/Approved\analysis' already exists.
Processing participant 2025_07_16_11_34_53...
Using image dimensions from imgSize column: (500, 500)


c:\Users\artem\Git\DeepEye_analyze\deepeye_analysis_package\preprocessing.py:155: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  group.FixStartEnd.iloc[0] = 'fix_end_carryover_inserted_start'
c:\Users\artem\Git\DeepEye_analyze\deepeye_analysi

Processing participant 2025_07_16_12_10_36...
Using image dimensions from imgSize column: (500, 500)


c:\Users\artem\Git\DeepEye_analyze\deepeye_analysis_package\preprocessing.py:155: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  group.FixStartEnd.iloc[0] = 'fix_end_carryover_inserted_start'
c:\Users\artem\Git\DeepEye_analyze\deepeye_analysi

Extracting fixations for participant 2025_07_17_11_47_19...
File does not exist: C:/Users/artem/Dropbox/Appliedwork/CognitiveSolutions/Projects/DeepEye/TechnicalReports/TechnicalReport1/Test_SkinAI/Approved\data\2025_07_17_11_47_19\2025_07_17_11_47_19_record.csv
Processing participant 2025_07_17_11_58_01...
Using image dimensions from imgSize column: (500, 500)


...

	Searching for valid interpolation windows
	Replace interpolation windows with Steffen interpolation
	2-Means clustering started for averaged signal
	Determining fixations based on clustering weight mean for averaged signal and separate eyes + 2*std


I2MC took 3.841564416885376s to finish!


c:\Users\artem\Git\DeepEye_analyze\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\artem\Git\DeepEye_analyze\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\artem\Git\DeepEye_analyze\.venv\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Processing participant 2025_09_25_07_24_19...
Using image dimensions from imgSize column: (500, 500)


c:\Users\artem\Git\DeepEye_analyze\deepeye_analysis_package\preprocessing.py:155: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  group.FixStartEnd.iloc[0] = 'fix_end_carryover_inserted_start'
c:\Users\artem\Git\DeepEye_analyze\deepeye_analysi

Combined data saved to C:/Users/artem/Dropbox/Appliedwork/CognitiveSolutions/Projects/DeepEye/TechnicalReports/TechnicalReport1/Test_SkinAI/Approved\analysis\allSubjects_SkinAI.csv
